In [1]:
import pandas as pd
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

def train_nids_model():
    # 1. Define column names
    columns = ["duration","protocol_type","service","flag","src_bytes","dst_bytes","land","wrong_fragment","urgent","hot",
               "num_failed_logins","logged_in","num_compromised","root_shell","su_attempted","num_root","num_file_creations",
               "num_shells","num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count","srv_count",
               "serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate",
               "srv_diff_host_rate","dst_host_count","dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
               "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
               "dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty"]

    # 2. Search Strategy (Keeping your current logic)
    user_home = os.path.expanduser("~")
    search_locations = [
        os.path.join(user_home, "Downloads", "archive (4)"),
        os.path.join(user_home, "OneDrive", "Documents", "archive (4)"),
        os.path.join(user_home, "Documents", "archive (4)"),
        "." 
    ]
    
    file_path = None
    for loc in search_locations:
        for name in ['KDDTrain+', 'KDDTrain+.txt']:
            check_path = os.path.join(loc, name)
            if os.path.exists(check_path):
                file_path = check_path
                break
        if file_path: break

    if file_path is None:
        print("❌ ERROR: Could not find 'KDDTrain+'")
        return

    print(f"✅ SUCCESS: Processing: {file_path}")
    
    # 3. Load and Process Data
    df = pd.read_csv(file_path, names=columns, header=None)
    df.drop(['difficulty'], axis=1, inplace=True)

    # --- IMPROVEMENT: ROBUST TARGET MAPPING ---
    # Explicitly map 'normal' and group all attack types into 'anomaly'
    df['label'] = df['label'].apply(lambda x: 'normal' if x.strip() == 'normal' else 'anomaly')

    # Categorical Encoding
    encoders = {}
    categorical_cols = ['protocol_type', 'service', 'flag']
    for col in categorical_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        encoders[col] = le

    # Split Data
    X = df.drop('label', axis=1)
    y = df['label']
    
    # --- IMPROVEMENT: FEATURE SCALING ---
    print("⚖️ Scaling numerical features...")
    scaler = StandardScaler()
    # We fit the scaler only on features (X)
    X_scaled = scaler.fit_transform(X)
    # Convert back to DataFrame to keep column names for the model
    X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

    # 4. Training
    print("🚀 Training Random Forest Core...")
    model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    model.fit(X_train, y_train)

    # 5. Save Artifacts (Added scaler.pkl)
    joblib.dump(model, 'nids_model.pkl')
    joblib.dump(encoders, 'encoders.pkl')
    joblib.dump(scaler, 'scaler.pkl') # IMPORTANT: Save the scaler for the dashboard
    joblib.dump(X.columns.tolist(), 'feature_columns.pkl')
    
    print(f"🎉 Model trained! Accuracy on test set: {model.score(X_test, y_test):.4f}")
    print("Files saved: nids_model.pkl, encoders.pkl, scaler.pkl, feature_columns.pkl")

if __name__ == "__main__":
    train_nids_model()

✅ SUCCESS: Processing: C:\Users\HP\OneDrive\Documents\archive (4)\KDDTrain+.txt
⚖️ Scaling numerical features...
🚀 Training Random Forest Core...
🎉 Model trained! Accuracy on test set: 0.9987
Files saved: nids_model.pkl, encoders.pkl, scaler.pkl, feature_columns.pkl


In [ ]:
!streamlit run app2.py